# 🕵️ Dia 44 — Trafegando inteligência com TAXII e marcando com TLP

**Investigador Cyber 90 Dias** · Fase 3 · Rank Jutsu `B`

> Baseado no *Day 44* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-3-cyber-threat-intelligence/day-44.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Três pedidos chegaram pelo `cluster-iscakit.json`: as quatro cooperativas afetadas, um grupo de analistas de fraude e um jornalista. Marta quer ajudar, sem ler o nome da Serra Azul no jornal. Rafa: *"Hoje a gente decide por escrito, e procura no JSON tudo o que não pode sair daqui."*

Neste notebook você vai:

1. **Escrever um cliente TAXII 2.1 mínimo com `urllib`** e percorrer o servidor público da MITRE (com plano B offline).
2. **Manter o registro de consultas** (*pull log*) em CSV.
3. **Preencher três registros de decisão de compartilhamento**.
4. **Marcar, expirar e limpar o bundle**, com as checagens do original em Python.
5. **Jogar o quiz "qual TLP?"**.

🐍 Só biblioteca padrão. O servidor da MITRE é lento: as células de rede têm *timeout* e caem no plano B.

## 1. Um cliente TAXII 2.1 mínimo

Três regras do protocolo, todas no código abaixo:

1. todo pedido leva o cabeçalho `Accept: application/taxii+json;version=2.1`;
2. filtros são parâmetros `match[campo]=valor` (os colchetes vão codificados na URL, que é o problema que o `-g` do curl resolve);
3. a resposta pagina com `more` e `next`.

Para o caso de não haver rede, o plano B é um **servidor simulado** que devolve respostas com o mesmo formato, inclusive um objeto revogado (`UNC2452`), como o original observou.

In [ ]:
import json                    # TAXII fala JSON
import csv                     # o registro de consultas vai em CSV
import urllib.parse            # para montar match[type]=... com os colchetes codificados
import urllib.request          # o cliente HTTP da biblioteca padrão
from datetime import datetime, timezone

BASE = "https://attack-taxii.mitre.org"
CABECALHO = {"Accept": "application/taxii+json;version=2.1"}   # sem isto, o servidor recusa
REGISTRO = []                  # uma linha por consulta: é o "pull log" do original

def agora():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

# ---------------- Plano B: um servidor TAXII de mentira, com o MESMO formato ----------------
ENTERPRISE = "x-mitre-collection--1f5f1533-f617-4ca8-9ab4-6a02367fa019"
SIMULADO_GRUPOS = [  # alguns intrusion-sets como o ATT&CK os entrega (inclui revogado e obsoleto)
    {"type": "intrusion-set", "name": "APT29", "id": "intrusion-set--899ce53f-13a0-479b-a0e4-67d46e241542"},
    {"type": "intrusion-set", "name": "UNC2452", "revoked": True, "id": "intrusion-set--d2c3e7ee-6c5a-4b1c-9bd8-2a1d3c9e4f10"},
    {"type": "intrusion-set", "name": "Sandworm Team", "id": "intrusion-set--381fcf73-60f6-4ab2-9991-6af3cbc35192"},
    {"type": "intrusion-set", "name": "Turla", "id": "intrusion-set--7a19ecb1-3c65-4de3-a230-993516aed6a6"},
    {"type": "intrusion-set", "name": "Grupo Obsoleto (exemplo)", "x_mitre_deprecated": True, "id": "intrusion-set--00000000-0000-4000-8000-000000000001"},
    {"type": "intrusion-set", "name": "OilRig", "id": "intrusion-set--4ca1929c-7d64-4aab-b849-badbfc0c760d"},
    {"type": "intrusion-set", "name": "APT28", "id": "intrusion-set--bef4c620-0787-42a8-a96d-b7eb6e85917c"},
]

def simulador(caminho, params):
    """Responde como o servidor da MITRE responderia, para os caminhos que usamos."""
    if caminho == "/taxii2/":
        return {"title": "ATT&CK TAXII 2.1 (SIMULADO)", "default": f"{BASE}/api/v21/",
                "api_roots": [f"{BASE}/api/v21/", f"{BASE}/api/v21/attack-11.0/", f"{BASE}/api/v21/attack-12.0/"]}
    if caminho.endswith("/collections/"):
        return {"collections": [
            {"id": "x-mitre-collection--90c00720-636b-4485-b342-8751d232bf09", "title": "ICS ATT&CK", "can_read": True, "can_write": False},
            {"id": ENTERPRISE, "title": "Enterprise ATT&CK", "can_read": True, "can_write": False},
            {"id": "x-mitre-collection--dac0d2d7-8653-445c-9bff-82f934c1e858", "title": "Mobile ATT&CK", "can_read": True, "can_write": False}]}
    if caminho.endswith("/objects/"):
        tipo = params.get("match[type]")
        objs = [o for o in SIMULADO_GRUPOS if tipo in (None, o["type"])]
        limite = int(params.get("limit", 5))
        inicio = int(params.get("next", 0))
        pagina = objs[inicio:inicio + limite]
        mais = inicio + limite < len(objs)
        resp = {"more": mais, "objects": pagina}
        if mais:
            resp["next"] = str(inicio + limite)
        return resp
    raise ValueError("caminho não simulado: " + caminho)
# --------------------------------------------------------------------------------------------

USAR_REDE = True       # mude para False para ir direto ao simulador (aula sem internet)

def taxii_get(caminho, **params):
    """GET num endpoint TAXII. Registra tudo no pull log. Cai no simulador se a rede falhar."""
    global USAR_REDE                                 # o "disjuntor" abaixo altera esta variável
    qs = urllib.parse.urlencode(params)              # match[type] vira match%5Btype%5D
    url = f"{BASE}{caminho}" + (f"?{qs}" if qs else "")
    fonte, erro = "rede", ""
    try:
        if not USAR_REDE:
            raise ConnectionError("rede desligada pelo usuário")
        req = urllib.request.Request(url, headers=CABECALHO)
        with urllib.request.urlopen(req, timeout=45) as r:   # o original avisa: 30 s ou mais por pedido
            dados = json.load(r)
    except Exception as e:
        USAR_REDE = False      # "disjuntor": depois da 1ª falha, não esperamos o timeout de novo
        dados, fonte, erro = simulador(caminho, params), "SIMULADO", type(e).__name__
    n = len(dados.get("objects", dados.get("collections", dados.get("api_roots", []))))
    REGISTRO.append({"utc": agora(), "url": url, "filtro": json.dumps(params), "fonte": fonte,
                     "objetos": n, "more": dados.get("more", ""), "erro": erro})
    return dados

d = taxii_get("/taxii2/")
print("Discovery:", d.get("title"), "| roots:", len(d.get("api_roots", [])), "| fonte:", REGISTRO[-1]["fonte"])

In [ ]:
# Coleções no root padrão (o "jq -r '.collections[] | [...] | @tsv'" do original)
cols = taxii_get("/api/v21/collections/")
for c in cols["collections"]:
    print(f"{c['id']}  {c['title']:<18} leitura={c['can_read']}  escrita={c['can_write']}")

## 2. Filtros, paginação e objetos revogados

Agora o laço que o `as_pages` da `taxii2-client` faz por você: pedir, ler `more`, devolver `next`. E o filtro que o **servidor não faz**: descartar `revoked` e `x_mitre_deprecated`.

In [ ]:
def todos_os_objetos(colecao, por_pagina=5, max_paginas=4, **filtros):
    """Percorre as páginas. max_paginas limita a aula (o servidor real tem centenas de grupos)."""
    params = {f"match[{k}]": v for k, v in filtros.items()}
    params["limit"] = por_pagina
    proximo, paginas = None, 0
    while paginas < max_paginas:
        p = dict(params, **({"next": proximo} if proximo else {}))
        pagina = taxii_get(f"/api/v21/collections/{colecao}/objects/", **p)
        paginas += 1
        yield from pagina.get("objects", [])
        if not pagina.get("more"):
            break
        proximo = pagina["next"]

grupos = list(todos_os_objetos(ENTERPRISE, type="intrusion-set"))
ativos = [g["name"] for g in grupos if not g.get("revoked") and not g.get("x_mitre_deprecated")]
descartados = [g["name"] for g in grupos if g.get("revoked") or g.get("x_mitre_deprecated")]

print(f"Recebidos: {len(grupos)} | ativos: {len(ativos)} | descartados: {descartados}")
print("Primeiros ativos:", ativos[:8])
print("\nℹ️  Para a contagem conciliada com o Dia 35, aumente max_paginas e por_pagina (o servidor real é lento).")

In [ ]:
# O registro de consultas (pull log): timestamp, URL, filtro, contagem, erros.
with open("taxii-pull-log.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(REGISTRO[0]))
    w.writeheader()
    w.writerows(REGISTRO)

for linha in REGISTRO:
    print(f"{linha['utc']}  {linha['fonte']:<8} objetos={linha['objetos']:<3} more={str(linha['more']):<5} {linha['url'][:90]}")
print("\n💾 taxii-pull-log.csv")

## 3. ➕ O bundle que vai sair da Vigia

Abaixo, uma versão do `cluster-iscakit.json` como ele estava **na pasta de trabalho**, antes da revisão. Ele tem **problemas plantados** de propósito, do tipo que aparece no mundo real. Não leia o JSON procurando: deixe as checagens do original acharem.

In [ ]:
BUNDLE = json.loads(r"""{
 "type": "bundle",
 "id": "bundle--9d1c7e52-2f4b-4a8e-b7c6-0e1f2a3b4c55",
 "objects": [
  {
   "type": "identity",
   "spec_version": "2.1",
   "id": "identity--3b0e3a63-1d5c-4f3e-9a51-6f2b8d0c7a10",
   "created": "2026-10-07T14:00:00.000Z",
   "modified": "2026-10-07T14:00:00.000Z",
   "name": "Vigia (MSSP fictício) - time de investigação",
   "identity_class": "organization"
  },
  {
   "type": "infrastructure",
   "spec_version": "2.1",
   "id": "infrastructure--8f6d2a4e-27b1-4c0e-9d3a-5e7f1b2c3d40",
   "created": "2026-10-07T14:00:00.000Z",
   "modified": "2026-10-07T14:00:00.000Z",
   "created_by_ref": "identity--3b0e3a63-1d5c-4f3e-9a51-6f2b8d0c7a10",
   "name": "Cluster IscaKit (Operação Isca)",
   "infrastructure_types": [
    "phishing"
   ],
   "description": "TODO: revisar com Rafa antes de enviar. String '// pescador' no JS do kit."
  },
  {
   "type": "indicator",
   "spec_version": "2.1",
   "id": "indicator--0b6f1c1e-6a4b-4d1f-8c55-1a2b3c4d5e01",
   "created": "2026-10-07T14:00:00.000Z",
   "modified": "2026-10-07T14:00:00.000Z",
   "created_by_ref": "identity--3b0e3a63-1d5c-4f3e-9a51-6f2b8d0c7a10",
   "name": "domain-name do cluster IscaKit: serraazul-pix[.]example",
   "description": "Origem: SMS recebidos pelos cooperados em 2026-10-05.",
   "indicator_types": [
    "malicious-activity"
   ],
   "pattern": "[domain-name:value = 'serraazul-pix.example']",
   "pattern_type": "stix",
   "valid_from": "2026-10-05T00:00:00Z",
   "valid_until": "2026-12-31T00:00:00Z",
   "confidence": 70,
   "object_marking_refs": [
    "marking-definition--f88d31f6-486f-44da-b317-01333bde0b82"
   ]
  },
  {
   "type": "indicator",
   "spec_version": "2.1",
   "id": "indicator--0b6f1c1e-6a4b-4d1f-8c55-1a2b3c4d5e02",
   "created": "2026-10-07T14:00:00.000Z",
   "modified": "2026-10-07T14:00:00.000Z",
   "created_by_ref": "identity--3b0e3a63-1d5c-4f3e-9a51-6f2b8d0c7a10",
   "name": "ipv4-addr do cluster IscaKit: 203[.]0[.]113[.]77",
   "description": "Origem: resolução DNS de serraazul-pix.example.",
   "indicator_types": [
    "malicious-activity"
   ],
   "pattern": "[ipv4-addr:value = '203.0.113.77']",
   "pattern_type": "stix",
   "valid_from": "2026-10-05T00:00:00Z",
   "valid_until": "2026-12-31T00:00:00Z",
   "confidence": 70,
   "object_marking_refs": [
    "marking-definition--f88d31f6-486f-44da-b317-01333bde0b82"
   ]
  },
  {
   "type": "indicator",
   "spec_version": "2.1",
   "id": "indicator--0b6f1c1e-6a4b-4d1f-8c55-1a2b3c4d5e03",
   "created": "2026-10-07T14:00:00.000Z",
   "modified": "2026-10-07T14:00:00.000Z",
   "created_by_ref": "identity--3b0e3a63-1d5c-4f3e-9a51-6f2b8d0c7a10",
   "name": "domain-name do cluster IscaKit: serraazul-seguranca[.]example",
   "description": "Origem: SAN do mesmo certificado TLS (crt.sh).",
   "indicator_types": [
    "malicious-activity"
   ],
   "pattern": "[domain-name:value = 'serraazul-seguranca.example']",
   "pattern_type": "stix",
   "valid_from": "2026-10-05T00:00:00Z",
   "valid_until": "2026-12-31T00:00:00Z",
   "confidence": 70,
   "object_marking_refs": [
    "marking-definition--f88d31f6-486f-44da-b317-01333bde0b82"
   ]
  },
  {
   "type": "indicator",
   "spec_version": "2.1",
   "id": "indicator--0b6f1c1e-6a4b-4d1f-8c55-1a2b3c4d5e04",
   "created": "2026-10-07T14:00:00.000Z",
   "modified": "2026-10-07T14:00:00.000Z",
   "created_by_ref": "identity--3b0e3a63-1d5c-4f3e-9a51-6f2b8d0c7a10",
   "name": "domain-name do cluster IscaKit: coopserra-azul[.]example",
   "description": "Origem: só semelhança de nome; ver C:\\Vigia\\casos\\VIGIA-2026-001\\notas-ana.txt",
   "indicator_types": [
    "malicious-activity"
   ],
   "pattern": "[domain-name:value = 'coopserra-azul.example']",
   "pattern_type": "stix",
   "valid_from": "2026-10-05T00:00:00Z",
   "valid_until": "2026-12-31T00:00:00Z",
   "confidence": 15,
   "object_marking_refs": [
    "marking-definition--f88d31f6-486f-44da-b317-01333bde0b82"
   ]
  },
  {
   "type": "indicator",
   "spec_version": "2.1",
   "id": "indicator--0b6f1c1e-6a4b-4d1f-8c55-1a2b3c4d5e05",
   "created": "2026-10-07T14:00:00.000Z",
   "modified": "2026-10-07T14:00:00.000Z",
   "created_by_ref": "identity--3b0e3a63-1d5c-4f3e-9a51-6f2b8d0c7a10",
   "name": "domain-name do cluster IscaKit: coopvale-pix[.]example",
   "description": "Origem: hash de corpo HTML do IscaKit (3 resultados). Confirmado por telefone com cooperado José, CPF 000.000.000-00 (fictício).",
   "indicator_types": [
    "malicious-activity"
   ],
   "pattern": "[domain-name:value = 'coopvale-pix.example']",
   "pattern_type": "stix",
   "valid_from": "2026-10-05T00:00:00Z",
   "valid_until": "2026-12-31T00:00:00Z",
   "confidence": 40,
   "object_marking_refs": [
    "marking-definition--f88d31f6-486f-44da-b317-01333bde0b82"
   ]
  },
  {
   "type": "indicator",
   "spec_version": "2.1",
   "id": "indicator--0b6f1c1e-6a4b-4d1f-8c55-1a2b3c4d5e06",
   "created": "2026-10-07T14:00:00.000Z",
   "modified": "2026-10-07T14:00:00.000Z",
   "created_by_ref": "identity--3b0e3a63-1d5c-4f3e-9a51-6f2b8d0c7a10",
   "name": "domain-name do cluster IscaKit: credminas-seguranca[.]example",
   "description": "Origem: favicon customizado (9 resultados).",
   "indicator_types": [
    "malicious-activity"
   ],
   "pattern": "[domain-name:value = 'credminas-seguranca.example']",
   "pattern_type": "stix",
   "valid_from": "2026-10-05T00:00:00Z",
   "confidence": 40,
   "object_marking_refs": [
    "marking-definition--f88d31f6-486f-44da-b317-01333bde0b82"
   ]
  },
  {
   "type": "relationship",
   "spec_version": "2.1",
   "id": "relationship--5a4b3c2d-1e0f-4a9b-8c7d-6e5f4a3b2c00",
   "created": "2026-10-07T14:00:00.000Z",
   "modified": "2026-10-07T14:00:00.000Z",
   "created_by_ref": "identity--3b0e3a63-1d5c-4f3e-9a51-6f2b8d0c7a10",
   "relationship_type": "indicates",
   "source_ref": "indicator--0b6f1c1e-6a4b-4d1f-8c55-1a2b3c4d5e01",
   "target_ref": "infrastructure--8f6d2a4e-27b1-4c0e-9d3a-5e7f1b2c3d40",
   "object_marking_refs": [
    "marking-definition--f88d31f6-486f-44da-b317-01333bde0b82"
   ],
   "confidence": 70
  },
  {
   "type": "relationship",
   "spec_version": "2.1",
   "id": "relationship--5a4b3c2d-1e0f-4a9b-8c7d-6e5f4a3b2c01",
   "created": "2026-10-07T14:00:00.000Z",
   "modified": "2026-10-07T14:00:00.000Z",
   "created_by_ref": "identity--3b0e3a63-1d5c-4f3e-9a51-6f2b8d0c7a10",
   "relationship_type": "indicates",
   "source_ref": "indicator--0b6f1c1e-6a4b-4d1f-8c55-1a2b3c4d5e02",
   "target_ref": "infrastructure--8f6d2a4e-27b1-4c0e-9d3a-5e7f1b2c3d40",
   "object_marking_refs": [
    "marking-definition--f88d31f6-486f-44da-b317-01333bde0b82"
   ],
   "confidence": 70
  },
  {
   "type": "relationship",
   "spec_version": "2.1",
   "id": "relationship--5a4b3c2d-1e0f-4a9b-8c7d-6e5f4a3b2c02",
   "created": "2026-10-07T14:00:00.000Z",
   "modified": "2026-10-07T14:00:00.000Z",
   "created_by_ref": "identity--3b0e3a63-1d5c-4f3e-9a51-6f2b8d0c7a10",
   "relationship_type": "indicates",
   "source_ref": "indicator--0b6f1c1e-6a4b-4d1f-8c55-1a2b3c4d5e03",
   "target_ref": "infrastructure--8f6d2a4e-27b1-4c0e-9d3a-5e7f1b2c3d40",
   "object_marking_refs": [
    "marking-definition--f88d31f6-486f-44da-b317-01333bde0b82"
   ],
   "confidence": 70
  },
  {
   "type": "relationship",
   "spec_version": "2.1",
   "id": "relationship--5a4b3c2d-1e0f-4a9b-8c7d-6e5f4a3b2c03",
   "created": "2026-10-07T14:00:00.000Z",
   "modified": "2026-10-07T14:00:00.000Z",
   "created_by_ref": "identity--3b0e3a63-1d5c-4f3e-9a51-6f2b8d0c7a10",
   "relationship_type": "indicates",
   "source_ref": "indicator--0b6f1c1e-6a4b-4d1f-8c55-1a2b3c4d5e04",
   "target_ref": "infrastructure--8f6d2a4e-27b1-4c0e-9d3a-5e7f1b2c3d40",
   "object_marking_refs": [
    "marking-definition--f88d31f6-486f-44da-b317-01333bde0b82"
   ],
   "confidence": 15
  },
  {
   "type": "relationship",
   "spec_version": "2.1",
   "id": "relationship--5a4b3c2d-1e0f-4a9b-8c7d-6e5f4a3b2c04",
   "created": "2026-10-07T14:00:00.000Z",
   "modified": "2026-10-07T14:00:00.000Z",
   "created_by_ref": "identity--3b0e3a63-1d5c-4f3e-9a51-6f2b8d0c7a10",
   "relationship_type": "indicates",
   "source_ref": "indicator--0b6f1c1e-6a4b-4d1f-8c55-1a2b3c4d5e05",
   "target_ref": "infrastructure--8f6d2a4e-27b1-4c0e-9d3a-5e7f1b2c3d40",
   "object_marking_refs": [
    "marking-definition--f88d31f6-486f-44da-b317-01333bde0b82"
   ],
   "confidence": 40
  },
  {
   "type": "relationship",
   "spec_version": "2.1",
   "id": "relationship--5a4b3c2d-1e0f-4a9b-8c7d-6e5f4a3b2c05",
   "created": "2026-10-07T14:00:00.000Z",
   "modified": "2026-10-07T14:00:00.000Z",
   "created_by_ref": "identity--3b0e3a63-1d5c-4f3e-9a51-6f2b8d0c7a10",
   "relationship_type": "indicates",
   "source_ref": "indicator--0b6f1c1e-6a4b-4d1f-8c55-1a2b3c4d5e06",
   "target_ref": "infrastructure--8f6d2a4e-27b1-4c0e-9d3a-5e7f1b2c3d40",
   "object_marking_refs": [
    "marking-definition--f88d31f6-486f-44da-b317-01333bde0b82"
   ],
   "confidence": 40
  }
 ]
}""")
print(len(BUNDLE["objects"]), "objetos carregados")

In [ ]:
# Checagem 1 do original (jq): objetos sem marcação, exceto marking-definition e identity.
def sem_marcacao(b):
    return [o["id"] for o in b["objects"]
            if o["type"] not in ("marking-definition", "identity") and not o.get("object_marking_refs")]

# Extra do passo 3: todo indicador precisa de valid_until.
def sem_expiracao(b):
    return [o["id"] for o in b["objects"] if o["type"] == "indicator" and "valid_until" not in o]

# Checagem 2 do original (o grep do passo 4), com o padrão ADAPTADO ao caso brasileiro.
import re
PADRAO_VAZAMENTO = re.compile(r"internal|corp|todo|password|senha|cpf|cooperado|[a-z]:\\|\\\\|@|notas-", re.I)

def todas_as_strings(x):
    """Equivalente do jq '.. | strings': percorre o JSON inteiro."""
    if isinstance(x, str):
        yield x
    elif isinstance(x, dict):
        for v in x.values():
            yield from todas_as_strings(v)
    elif isinstance(x, list):
        for v in x:
            yield from todas_as_strings(v)

def vazamentos(b):
    return [s for s in todas_as_strings(b) if PADRAO_VAZAMENTO.search(s)]

print("Sem marcação :", sem_marcacao(BUNDLE))
print("Sem expiração:", sem_expiracao(BUNDLE))
print("Possíveis vazamentos (leia CADA um):")
for s in vazamentos(BUNDLE):
    print("   ⚠️ ", s)

### 🔧 Corrija

A célula seguinte aplica as correções **que um humano decidiu** depois de ler cada ocorrência (e registra os falsos positivos aceitos, como "corp" dentro de "corpo"): marcação e expiração em tudo, e reescrita das descrições que citavam caminho interno, nome e CPF de cooperado e um TODO. Leia as correções antes de rodar: nenhuma delas é automática "por regex", porque apagar às cegas pode destruir a evidência de origem.

In [ ]:
import copy
LIMPO = copy.deepcopy(BUNDLE)
TLP_AMBER = "marking-definition--f88d31f6-486f-44da-b317-01333bde0b82"

REESCRITAS = {   # id → nova descrição, decidida pelo analista
    "indicator--0b6f1c1e-6a4b-4d1f-8c55-1a2b3c4d5e04":
        "Origem: só semelhança de nome com a cooperativa; registrado em 2026-10-03; sem kit observado.",
    "indicator--0b6f1c1e-6a4b-4d1f-8c55-1a2b3c4d5e05":
        "Origem: hash de corpo HTML do IscaKit (3 resultados) e DNS passivo.",
    "infrastructure--8f6d2a4e-27b1-4c0e-9d3a-5e7f1b2c3d40":
        "Páginas de captura de credenciais do IscaKit. Artefato textual do kit: string '// pescador' no JS.",
}
for o in LIMPO["objects"]:
    if o["id"] in REESCRITAS:
        o["description"] = REESCRITAS[o["id"]]
    if o["type"] not in ("marking-definition", "identity"):
        o.setdefault("object_marking_refs", [TLP_AMBER])
    if o["type"] == "indicator":
        o.setdefault("valid_until", "2026-12-31T00:00:00Z")

# O grep acha "corp" dentro de "corpo" e "cooperado" dentro de "cooperados": FALSOS POSITIVOS.
# O original manda ler cada ocorrência. Depois de lidas, registramos quais foram aceitas e por quê.
REVISADOS = {
    "Origem: SMS recebidos pelos cooperados em 2026-10-05.": "fala dos cooperados em geral, sem identificar ninguém",
    "Origem: hash de corpo HTML do IscaKit (3 resultados) e DNS passivo.": "'corpo' casou com 'corp'",
}
print("Sem marcação :", sem_marcacao(LIMPO))      # o original espera []
print("Sem expiração:", sem_expiracao(LIMPO))
v = [s for s in vazamentos(LIMPO) if s not in REVISADOS]
print("Vazamentos   :", v if v else "clean")       # o original imprime "clean"
for s, motivo in REVISADOS.items():
    print(f"   (falso positivo revisado: {motivo})")

## 4. ➕ Três destinatários, três registros de decisão

Preencha o formulário (os valores padrão seguem a tabela da Prática BR). A célula aplica a decisão aos dados: troca a marcação de cada objeto pelo TLP escolhido, filtra pela confiança mínima, e grava um bundle e um registro por destinatário. Para **TLP:GREEN** usamos a marcação predefinida do STIX 2.1; para **TLP:CLEAR** e **AMBER+STRICT**, lembre-se de que o STIX 2.1 não as predefine (ver Conceito) e o notebook se recusa a gerar o bundle.

In [ ]:
#@title Registros de decisão { display-mode: "form" }
coop_tlp = "TLP:AMBER"  #@param ["TLP:RED", "TLP:AMBER+STRICT", "TLP:AMBER", "TLP:GREEN", "TLP:CLEAR"]
coop_conf_min = 0  #@param {type:"integer"}
coop_por_que_nao_clear = "Pivôs da Vigia não publicados; domínios ainda no ar; publicar avisaria o operador do kit."  #@param {type:"string"}
coop_por_que_nao_mais_restrito = "As cooperativas precisam repassar aos fornecedores de SOC delas para bloquear."  #@param {type:"string"}

comunidade_tlp = "TLP:GREEN"  #@param ["TLP:RED", "TLP:AMBER+STRICT", "TLP:AMBER", "TLP:GREEN", "TLP:CLEAR"]
comunidade_conf_min = 40  #@param {type:"integer"}
comunidade_por_que_nao_clear = "Derrubada em curso; publicação ampla pode acelerar a troca de infraestrutura."  #@param {type:"string"}
comunidade_por_que_nao_mais_restrito = "O valor está em alertar outras cooperativas antes de serem alvo."  #@param {type:"string"}

jornalista_tlp = "nada"  #@param ["nada", "TLP:CLEAR"]
jornalista_motivo = "A Vigia não é fonte; a comunicação pública é da cooperativa, sem indicadores."  #@param {type:"string"}
aprovado_por = "Marta Quintão (Serra Azul) e Rafa (Vigia)"  #@param {type:"string"}

PREDEFINIDAS = {"TLP:GREEN": "marking-definition--34098fce-860f-48ae-8e50-ebd3cc5e41da",
                "TLP:AMBER": "marking-definition--f88d31f6-486f-44da-b317-01333bde0b82",
                "TLP:RED":   "marking-definition--5e57c739-391a-4eb3-b6be-7d15ca92d5ed"}

def preparar(nome, tlp, conf_min, nao_clear, nao_restrito):
    registro = (f"# Registro de decisão de compartilhamento — {nome}\n"
                f"Conteúdo: cluster-iscakit.json (indicadores com confiança ≥ {conf_min})\n"
                "Fontes dentro dele: SMS dos cooperados (interno), pivôs passivos da Vigia (não publicados)\n"
                "Algum objeto pode identificar vítima, pessoa ou a nossa organização? não, após a revisão do passo 3\n"
                "Compartilhar pode alertar o ator ou queimar uma fonte? sim se público; por isso não CLEAR\n"
                f"Destinatários: {nome}\nTLP escolhido: {tlp}\n"
                f"Por que não CLEAR: {nao_clear}\nPor que não um nível mais restrito: {nao_restrito}\n"
                "Expiração / revisão: valid_until = 2026-12-31 em todos os indicadores\n"
                f"Aprovado por: {aprovado_por}\n")
    if tlp not in PREDEFINIDAS:
        print(f"⛔ {nome}: {tlp} não é predefinido no STIX 2.1. Use a definição de marcação TLP 2.0 da OASIS "
              "e confirme que o destinatário a entende. Bundle NÃO gerado.")
        return registro, None
    b = copy.deepcopy(LIMPO)
    manter = {o["id"] for o in b["objects"] if o["type"] != "indicator" or o["confidence"] >= conf_min}
    b["objects"] = [o for o in b["objects"]
                    if o["id"] in manter and (o["type"] != "relationship" or o["source_ref"] in manter)]
    for o in b["objects"]:
        if "object_marking_refs" in o:
            o["object_marking_refs"] = [PREDEFINIDAS[tlp]]
    return registro, b

import pathlib
pathlib.Path("compartilhamento").mkdir(exist_ok=True)
for chave, nome, tlp, cmin, a, b_ in [
        ("cooperativas", "Times de risco das 4 cooperativas afetadas", coop_tlp, coop_conf_min, coop_por_que_nao_clear, coop_por_que_nao_mais_restrito),
        ("comunidade", "Grupo de analistas de fraude de cooperativas", comunidade_tlp, comunidade_conf_min, comunidade_por_que_nao_clear, comunidade_por_que_nao_mais_restrito)]:
    reg, bun = preparar(nome, tlp, cmin, a, b_)
    pathlib.Path(f"compartilhamento/decisao-{chave}.md").write_text(reg, encoding="utf-8")
    if bun:
        pathlib.Path(f"compartilhamento/bundle-{chave}.json").write_text(json.dumps(bun, ensure_ascii=False, indent=2), encoding="utf-8")
        n = sum(1 for o in bun["objects"] if o["type"] == "indicator")
        print(f"✅ {chave}: {tlp}, {n} indicadores, sem marcação={sem_marcacao(bun)}")

pathlib.Path("compartilhamento/decisao-jornalista.md").write_text(
    f"# Registro de decisão — jornalista\nDecisão: {jornalista_tlp}\nMotivo: {jornalista_motivo}\nAprovado por: {aprovado_por}\n",
    encoding="utf-8")
print("✅ jornalista:", jornalista_tlp, "—", jornalista_motivo)

## 5. 🎮 Quiz: qual TLP?

Responda com o número do rótulo: **1** RED · **2** AMBER+STRICT · **3** AMBER · **4** GREEN · **5** CLEAR. Algumas situações aceitam discussão; o gabarito traz a resposta mais defensável e o porquê.

In [ ]:
ROTULOS = {"1": "TLP:RED", "2": "TLP:AMBER+STRICT", "3": "TLP:AMBER", "4": "TLP:GREEN", "5": "TLP:CLEAR"}
CASOS = [
    ("Alerta público da cooperativa aos 40 mil cooperados: 'não clique em links de SMS sobre Pix bloqueado'.", "5",
     "Feito para ser publicado. Sem indicadores técnicos sensíveis."),
    ("Lista de domínios do IscaKit para a comunidade fechada de analistas de fraude.", "4",
     "Comunidade sim, público não: a derrubada está em curso."),
    ("Bundle completo para as 4 cooperativas, que vão repassar ao SOC terceirizado de cada uma.", "3",
     "AMBER permite clientes/necessidade de saber. AMBER+STRICT impediria o repasse ao SOC terceirizado."),
    ("Relato à Marta e ao Dr. Hélio de que um funcionário da Serra Azul pode ter clicado no link.", "1",
     "Só os destinatários nomeados, numa conversa. Envolve pessoa identificável e questão trabalhista."),
    ("Análise que só a equipe interna de segurança da CoopVale pode ler, sem repassar a fornecedores.", "2",
     "Só a organização do destinatário: é exatamente o AMBER+STRICT do TLP 2.0."),
]
MODO_JOGO = True

acertos = 0
for i, (caso, certo, porque) in enumerate(CASOS, start=1):
    print(f"\n[{i}/{len(CASOS)}] {caso}")
    if MODO_JOGO:
        r = ""
        while r not in ROTULOS:
            r = input("   1 RED · 2 AMBER+STRICT · 3 AMBER · 4 GREEN · 5 CLEAR: ").strip()
        acertos += (r == certo)
        print("   ✅" if r == certo else f"   ❌ (você: {ROTULOS[r]})")
    print(f"   Gabarito: {ROTULOS[certo]} — {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(CASOS)}")

In [ ]:
try:
    from google.colab import files
    for nome in ("taxii-pull-log.csv", "compartilhamento/decisao-cooperativas.md",
                 "compartilhamento/decisao-comunidade.md", "compartilhamento/decisao-jornalista.md"):
        files.download(nome)
except ImportError:
    print("(Fora do Colab: arquivos em ./taxii-pull-log.csv e ./compartilhamento/)")

## ✅ Antes de fechar o Dia 44

- [ ] `taxii-pull-log.csv` com discovery, coleções, consultas filtradas e paginação
- [ ] Você explica por que `UNC2452` veio na resposta e foi descartado
- [ ] As duas checagens do original devolvem `[]` e `clean` no bundle limpo
- [ ] Três registros de decisão em `compartilhamento/`
- [ ] (Original) contagem de intrusion sets ativos conciliada com o Dia 35

**Amanhã (Dia 45):** a pergunta muda de "quem pode ver?" para "o que está acontecendo de fato?". Você monta uma matriz ACH para o IscaKit nas cinco cooperativas: um operador, vários compradores do mesmo kit, ou um imitador?

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>